## **Caching in Langchain RAG**

**1.Exact-Match Caching** : Uses hash-based lookup - only identical queries hit the cache

**2.Semantic Caching** : Uses embedding similarity- semantically similar queries can hit the cache

**Prerequisites:** : Docker must be installed and running Redis

In [28]:
import os,redis,hashlib,json
import numpy as np 
from typing import List, Optional, Tuple 

from langchain_openai import ChatOpenAI , OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pydantic import PrivateAttr

In [29]:
import subprocess, time 

# Remove existing container if present and start a fresh
subprocess.run(['docker' , 'rm' , '-f' ,'redis-cache'] , capture_output=True)
subprocess.run(['docker' , 'run' ,'-d' , '--name' , 'redis-cache' , '-p' , '6379:6379' , 'redis'] , check=True)

time.sleep(2)
print("Redis container started")

Redis container started


In [30]:
# Initlaize the redis cache
cache = redis.Redis(host="localhost" , port = 6379)

#### **Part-1 :- Exact Matching Caching**

In [31]:
class CachedRetriever(BaseRetriever):
    """ 
        A langchain retriever that wraps another retriever with redis caching.
    """
    _base_retriever : any = PrivateAttr()
    _cache_ttl : int = PrivateAttr(default=3600)

    def __init__(self,base_retriever,cache_ttl : int = 3600 , **kwargs):
        super().__init__(**kwargs)
        self._base_retriever = base_retriever
        self._cache_ttl = cache_ttl

    def _get_relevant_documents(self , query : str)-> List[Document]:
        # Create a deterministic hash of the query
        query_hash = hashlib.sha256(query.encode('utf-8')).hexdigest()

        # Check fast cache(Redis)
        cached_result = cache.get(query_hash)
        if cached_result:
            print("Cache Hit! Skipping the vector search")
            docs_data = json.loads(cached_result)
            return [Document(page_content=d['content'] , metadata = d['metadata']) for d in docs_data]
        
        # Cache miss : Perform the expensive vector search
        print("Cache Miss, Performing the vector search ... ")
        results = self._base_retriever.invoke(query)

        # Serialize and store in cache with TTL
        docs_data = [{'content' : doc.page_content , 'metadata' : doc.metadata} for doc in results]
        cache.setex(query_hash , self._cache_ttl , json.dumps(docs_data))

        return results 

In [32]:
class SemanticCachedRetriever(BaseRetriever):
    """ 
         A retriever that uses embedding similarity for semantic caching.
         Similar queries can hit the cache even if worded differently.
    """
    _base_retriever : any = PrivateAttr()
    _embeddings : any = PrivateAttr()
    _similarity_threshold : float = PrivateAttr(default=0.85)
    _cache_embeddings : List[np.ndarray] = PrivateAttr(default_factory=list)
    _cache_results : List[List[Document]] = PrivateAttr(default_factory=list)
    _cache_queries : List[str] = PrivateAttr(default_factory=list)


    def __init__(self , base_retriever , embeddings , similarity_threshold : float = 0.85 , **kwargs):
        super().__init__(**kwargs)
        self._base_retriever = base_retriever
        self._similarity_threshold = similarity_threshold
        self._embeddings = embeddings
        

    def _cosine_similarity(self , a: np.ndarray , b : np.ndarray) -> float:
        """Compute cosine similarity between two vectors."""
        return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))
    

    def _find_similar_cached(self , query_embedding : np.ndarray) -> Optional[Tuple[List[Document] , str , float]]:
        best_similarity = 0.0 
        best_match = None 
        best_query = None 

        for i , cached_emb in enumerate(self._cache_embeddings):
            similarity = self._cosine_similarity(query_embedding , cached_emb)
            if similarity > best_similarity:
                best_similarity = similarity
                best_match = self._cache_results[i]
                best_query = self._cache_queries[i]

        if best_similarity >= self._similarity_threshold:
            return best_match , best_query , best_similarity
        return None 
    
    def _get_relevant_documents(self , query : str) -> List[Document]:
        # First get the embedding for the query
        query_embedding = np.array(self._embeddings.embed_query(query))

        # Check for semantically similary cache query
        cache_hit = self._find_similar_cached(query_embedding)
        if cache_hit:
            docs,original_query,similarity = cache_hit
            print(f"Semantic Cache hit. Similarity : {similarity}")
            print(f"Matched query : {original_query}")
            return docs 
        
        # Cache miss 
        print("Semantic Cache Miss. Perorming vector search...")
        results = self._base_retriever.invoke(query)

        # Store the cache
        self._cache_embeddings.append(query_embedding)
        self._cache_results.append(results)
        self._cache_queries.append(query)

    def clear_cache(self):
        """Clear the semantic cache."""
        self._cache_embeddings = []
        self._cache_results = []
        self._cache_queries = []

In [33]:
# Create sample documents for the demo
sample_text = """
Artificial Intelligence (AI) is transforming industries worldwide.
Machine learning models can analyze vast amounts of data to find patterns.
Deep learning uses neural networks with multiple layers to learn representations.
Natural Language Processing (NLP) enables computers to understand human language.
Retrieval-Augmented Generation (RAG) combines retrieval systems with language models.
RAG helps reduce hallucinations by grounding responses in retrieved documents.
Vector databases store embeddings for efficient similarity search.
Caching can significantly improve RAG system performance by avoiding redundant searches.
"""

# Save sample text to a file
with open("sample_docs.txt", "w") as f:
    f.write(sample_text)

print("Sample documents created.")

Sample documents created.


In [34]:
# Initialize LLM and embeddings
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0 , api_key=os.getenv("OPEN_AI_API"))
embeddings = OpenAIEmbeddings(api_key=os.getenv("OPEN_AI_API"))

# Load and split documents
loader = TextLoader("sample_docs.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
texts = text_splitter.split_documents(docs)

print(f"Created {len(texts)} text chunks")

Created 4 text chunks


In [35]:
# Create vector store and base retriever
vectorstore = FAISS.from_documents(texts, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Wrap with caching
cached_retriever = CachedRetriever(base_retriever, cache_ttl=3600)

print("Vector store and cached retriever initialized.")

Vector store and cached retriever initialized.


In [36]:
# Create the RAG chain with cached retriever using LCEL
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": cached_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created with cached retriever.")

RAG chain created with cached retriever.


In [37]:
# Clear any existing cache for demo purposes
query = "What is RAG and how does it help?"
query_hash = hashlib.sha256(query.encode('utf-8')).hexdigest()
cache.delete(query_hash)

print("Cache cleared for demo query.")

Cache cleared for demo query.


In [38]:
# First query - Cache Miss (performs vector search)
print("=== First Query (Cache Miss) ===")
response = rag_chain.invoke(query)
print(f"\nAnswer: {response}")

=== First Query (Cache Miss) ===
Cache Miss, Performing the vector search ... 


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2164\344183929.py:30: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  cache.setex(query_hash , self._cache_ttl , json.dumps(docs_data))



Answer: Retrieval-Augmented Generation (RAG) is a system that combines retrieval mechanisms with language models. It helps reduce hallucinations in generated responses by grounding them in retrieved documents, ensuring that the information provided is more accurate and reliable.


### **Comparison - The Power of Semantic Caching**

In [39]:
# Define our test queries
original_query = "What is RAG and how does it help?"
similar_query = "Tell me about RAG and its benefits"

print(f"Original query: \"{original_query}\"")
print(f"Similar query:  \"{similar_query}\"")
print(f"\nThese ask the same thing but with different wording.")

Original query: "What is RAG and how does it help?"
Similar query:  "Tell me about RAG and its benefits"

These ask the same thing but with different wording.


In [40]:
# Test exact-match cache with the similar query
print("=== Exact-Match Cache Test ===")
print(f"Query: \"{similar_query}\"\n")

# This will miss because the hash is different
_ = cached_retriever.invoke(similar_query)

print("\n❌ Exact-match cache missed - had to perform vector search again!")

=== Exact-Match Cache Test ===
Query: "Tell me about RAG and its benefits"

Cache Miss, Performing the vector search ... 

❌ Exact-match cache missed - had to perform vector search again!


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2164\344183929.py:30: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  cache.setex(query_hash , self._cache_ttl , json.dumps(docs_data))


In [41]:
# Create semantic cached retriever
semantic_retriever = SemanticCachedRetriever(
    base_retriever, 
    embeddings,
    similarity_threshold=0.85
)

# First, prime the cache with the original query
print("=== Priming Semantic Cache ===")
print(f"Query: \"{original_query}\"\n")
_ = semantic_retriever.invoke(original_query)

=== Priming Semantic Cache ===
Query: "What is RAG and how does it help?"

Semantic Cache Miss. Perorming vector search...


In [42]:
# Now test with the semantically similar query
print("=== Semantic Cache Test ===")
print(f"Query: \"{similar_query}\"\n")
_ = semantic_retriever.invoke(similar_query)

print("\n✓ Semantic cache hit - no vector search needed!")

=== Semantic Cache Test ===
Query: "Tell me about RAG and its benefits"

Semantic Cache hit. Similarity : 0.9471071538211745
Matched query : What is RAG and how does it help?

✓ Semantic cache hit - no vector search needed!


In [43]:
# Try another semantically similar query
third_query = "Explain what RAG is and why it's useful"
print("=== Another Semantic Cache Test ===")
print(f"Query: \"{third_query}\"\n")
_ = semantic_retriever.invoke(third_query)

print("\n✓ Another hit! The cache recognizes this is asking the same thing.")

=== Another Semantic Cache Test ===
Query: "Explain what RAG is and why it's useful"

Semantic Cache hit. Similarity : 0.9544918063267006
Matched query : What is RAG and how does it help?

✓ Another hit! The cache recognizes this is asking the same thing.


In [44]:
# Test with a genuinely different query - should miss
different_query = "What are vector databases?"
print("=== Different Topic Query ===")
print(f"Query: \"{different_query}\"\n")
_ = semantic_retriever.invoke(different_query)

print("\nThis query is about a different topic, so it correctly misses the cache.")

=== Different Topic Query ===
Query: "What are vector databases?"

Semantic Cache Miss. Perorming vector search...

This query is about a different topic, so it correctly misses the cache.


In [46]:
# Cleanup
import os
import subprocess

os.remove("sample_docs.txt")
subprocess.run(["docker", "rm", "-f", "redis-cache"], capture_output=True)
print("Cleanup complete. Redis container stopped.")

Cleanup complete. Redis container stopped.
